# 02 - Enriquecimiento y unificacion

Objetivos de este notebook:
- Integrar Taxi Zones para origen (PU) y destino (DO).
- Unificar viajes de Yellow y Green en una estructura comun.
- Normalizar catalogos operativos (vendor, rate code, payment type, trip type, store and forward).
- Publicar tablas curadas en Snowflake.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from urllib.request import Request, urlopen
import datetime
import glob
import os

In [2]:
def log_step(message):
    timestamp = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
    print(f'[{timestamp}Z] {message}')


def build_spark(app_name):
    jars_dir = os.environ.get('SPARK_JARS_DIR', '/home/jovyan/jars')
    jar_paths = sorted(glob.glob(os.path.join(jars_dir, '*.jar')))
    builder = (
        SparkSession.builder
        .appName(app_name)
        .config('spark.sql.session.timeZone', 'UTC')
    )
    if jar_paths:
        jars_csv = ','.join(jar_paths)
        classpath = ':'.join(jar_paths)
        log_step(f'Loading JARs: {jar_paths}')
        builder = (
            builder
            .config('spark.jars', jars_csv)
            .config('spark.driver.extraClassPath', classpath)
            .config('spark.executor.extraClassPath', classpath)
        )
    else:
        log_step(f'No JARs found in {jars_dir}')
    return builder.getOrCreate()


def assert_snowflake_connector(spark):
    class_names = ['net.snowflake.spark.snowflake.DefaultSource', 'snowflake.DefaultSource']
    for class_name in class_names:
        try:
            spark.sparkContext._jvm.java.lang.Class.forName(class_name)
            log_step(f'Snowflake connector detected: {class_name}')
            return
        except Exception:
            continue
    raise RuntimeError('Snowflake connector not found. Run `bash scripts/download_snowflake_jars.sh` from the repo root, restart the container, and rerun this notebook.')


app = build_spark('02_enriquecimiento_y_unificacion')
assert_snowflake_connector(app)

# Credenciales Snowflake desde variables de ambiente
curated_schema = os.environ.get('SF_CURATED_SCHEMA', 'CURATED')

sf_option = {
    'sfURL': os.environ['SF_HOST'],
    'sfUser': os.environ['SF_USER'],
    'sfPassword': os.environ['SF_PASSWORD'],
    'sfDatabase': os.environ['SF_DATABASE'],
    'sfSchema': os.environ.get('SF_RAW_SCHEMA', 'raw'),
    'sfWarehouse': os.environ.get('SF_WAREHOUSE', ''),
    'sfRole': os.environ.get('SF_ROLE', ''),
}

def write_snowflake_table(df, dbtable, mode='overwrite', preactions=None):
    writer = (
        df.write.format('snowflake')
        .options(**sf_option)
        .option('dbtable', dbtable)
        .mode(mode)
    )
    if preactions:
        writer = writer.option('preactions', preactions)
    writer.save()

log_step(f'Notebook 02 ready: curated_schema={curated_schema}')

[2026-04-05 22:02:13Z] Loading JARs: ['/home/jovyan/jars/snowflake-jdbc-3.28.0.jar', '/home/jovyan/jars/spark-snowflake_2.12-3.1.8.jar']
[2026-04-05 22:02:15Z] Snowflake connector detected: net.snowflake.spark.snowflake.DefaultSource
[2026-04-05 22:02:15Z] Notebook 02 ready: curated_schema=CURATED


In [3]:
# Lectura de tablas RAW generadas en el notebook 01
def normalize_columns(df):
    return df.toDF(*[c.lower() for c in df.columns])

df_yellow_raw = normalize_columns(
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'RAW.YELLOW_TRIPS')
    .load()
)

df_green_raw = normalize_columns(
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'RAW.GREEN_TRIPS')
    .load()
)

yellow_rows = df_yellow_raw.count()
green_rows = df_green_raw.count()
log_step(f'Loaded RAW tables: yellow_rows={yellow_rows}, green_rows={green_rows}')

[2026-04-05 22:04:03Z] Loaded RAW tables: yellow_rows=25183429, green_rows=3083323


In [4]:
# Estandarizacion de timestamps y claves operativas para unificar tipos de taxi
def base_columns(df):
    columns = set(df.columns)

    def existing_col(*names):
        for name in names:
            if name in columns:
                return F.col(name)
        return F.lit(None)

    pickup_col = existing_col('tpep_pickup_datetime', 'lpep_pickup_datetime', 'pickup_datetime')
    dropoff_col = existing_col('tpep_dropoff_datetime', 'lpep_dropoff_datetime', 'dropoff_datetime')
    vendor_col = existing_col('vendorid', 'vendor_id').cast('int')
    rate_col = existing_col('ratecodeid', 'rate_code', 'rate_code_id').cast('int')
    payment_col = existing_col('payment_type', 'payment_type_code').cast('int')
    trip_type_col = existing_col('trip_type', 'trip_type_code').cast('int')
    store_and_fwd_col = existing_col('store_and_fwd_flag', 'store_and_fwd_flag_norm')

    return (
        df
        .withColumn('pickup_datetime_utc', pickup_col.cast('timestamp'))
        .withColumn('dropoff_datetime_utc', dropoff_col.cast('timestamp'))
        .withColumn('vendor_id', vendor_col)
        .withColumn('rate_code_id', rate_col)
        .withColumn('payment_type_code', payment_col)
        .withColumn('trip_type_code', trip_type_col)
        .withColumn('store_and_fwd_flag_norm', F.upper(F.trim(store_and_fwd_col.cast('string'))))
    )

In [5]:
# Se aplica el mismo esquema de negocio para yellow y green
df_yellow = base_columns(df_yellow_raw).withColumn('taxi_type', F.lit('yellow'))
df_green = base_columns(df_green_raw).withColumn('taxi_type', F.lit('green'))

unified_cols = [
    'trip_nk', 'run_id', 'service_type', 'taxi_type',
    'source_year', 'source_month', 'ingested_at_utc', 'source_path',
    'pickup_datetime_utc', 'dropoff_datetime_utc',
    'pulocationid', 'dolocationid',
    'vendor_id', 'rate_code_id', 'payment_type_code', 'trip_type_code', 'store_and_fwd_flag_norm',
    'passenger_count', 'trip_distance',
    'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
    'improvement_surcharge', 'congestion_surcharge', 'airport_fee', 'total_amount'
]

for c in unified_cols:
    if c not in df_yellow.columns:
        df_yellow = df_yellow.withColumn(c, F.lit(None))
    if c not in df_green.columns:
        df_green = df_green.withColumn(c, F.lit(None))

df_trips_unified = df_yellow.select(*unified_cols).unionByName(df_green.select(*unified_cols))

In [6]:
# Integracion Taxi Zones desde lookup oficial TLC / NYC Open Data
def download_reference_file(url_candidates, filename):
    local_dir = os.environ.get('LOCAL_REFERENCE_DIR', '/tmp/nyc_taxi_reference')
    os.makedirs(local_dir, exist_ok=True)
    local_path = os.path.join(local_dir, filename)

    if os.path.exists(local_path):
        log_step(f'Using cached reference file: {local_path}')
        return local_path

    errors = []
    for url in url_candidates:
        if not url:
            continue
        try:
            log_step(f'Downloading reference file: {url}')
            request = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urlopen(request) as response, open(local_path, 'wb') as output_file:
                output_file.write(response.read())
            return local_path
        except Exception as exc:
            errors.append(f'{url} -> {str(exc)[:160]}')
            log_step(f'Reference download failed: {url} | {str(exc)[:160]}')

    raise RuntimeError('Could not download Taxi Zone lookup from any configured source. Tried: ' + ' || '.join(errors))

zones_url_candidates = [
    os.environ.get('TAXI_ZONE_PATH', ''),
    'https://data.cityofnewyork.us/api/views/8meu-9t5y/rows.csv?accessType=DOWNLOAD',
    'https://s3.amazonaws.com/nyc-tlc/misc/taxi+_zone_lookup.csv',
    'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv',
]
zones_path = download_reference_file(zones_url_candidates, 'taxi_zone_lookup.csv')

df_zones = (
    app.read.option('header', True).option('inferSchema', True).csv(zones_path)
    .withColumn('locationid', F.col('LocationID').cast('int'))
    .withColumn('borough', F.trim(F.col('Borough')))
    .withColumn('zone', F.trim(F.col('Zone')))
    .withColumn('service_zone', F.trim(F.col('service_zone')))
    .dropDuplicates(['locationid'])
)

df_zones_pu = (
    df_zones
    .select(
        F.col('locationid').alias('pu_location_id'),
        F.col('borough').alias('pu_borough'),
        F.col('zone').alias('pu_zone'),
        F.col('service_zone').alias('pu_service_zone')
    )
)

df_zones_do = (
    df_zones
    .select(
        F.col('locationid').alias('do_location_id'),
        F.col('borough').alias('do_borough'),
        F.col('zone').alias('do_zone'),
        F.col('service_zone').alias('do_service_zone')
    )
)

[2026-04-05 22:04:03Z] Downloading reference file: https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv


In [7]:
# Normalizacion de catalogos (codigos a descripciones)
catalog_vendor = [
    (1, 'Creative Mobile Technologies'),
    (2, 'VeriFone Inc.'),
    (6, 'Myle Technologies Inc.')
]

catalog_ratecode = [
    (1, 'Standard rate'),
    (2, 'JFK'),
    (3, 'Newark'),
    (4, 'Nassau or Westchester'),
    (5, 'Negotiated fare'),
    (6, 'Group ride')
]

catalog_payment_type = [
    (1, 'Credit card'),
    (2, 'Cash'),
    (3, 'No charge'),
    (4, 'Dispute'),
    (5, 'Unknown'),
    (6, 'Voided trip')
]

catalog_trip_type = [
    (1, 'Street-hail'),
    (2, 'Dispatch')
]

catalog_store_and_fwd = [
    ('Y', 'Stored and forwarded'),
    ('N', 'Not a stored trip')
]

df_cat_vendor = app.createDataFrame(catalog_vendor, ['vendor_id', 'vendor_desc'])
df_cat_rate = app.createDataFrame(catalog_ratecode, ['rate_code_id', 'rate_code_desc'])
df_cat_payment = app.createDataFrame(catalog_payment_type, ['payment_type_code', 'payment_type_desc'])
df_cat_trip = app.createDataFrame(catalog_trip_type, ['trip_type_code', 'trip_type_desc'])
df_cat_saf = app.createDataFrame(catalog_store_and_fwd, ['store_and_fwd_flag_norm', 'store_and_fwd_desc'])

In [8]:
df_trips_enriched = (
    df_trips_unified
    .withColumn('pulocationid', F.col('pulocationid').cast('int'))
    .withColumn('dolocationid', F.col('dolocationid').cast('int'))
    .join(df_zones_pu, F.col('pulocationid') == F.col('pu_location_id'), 'left')
    .join(df_zones_do, F.col('dolocationid') == F.col('do_location_id'), 'left')
    .join(df_cat_vendor, on='vendor_id', how='left')
    .join(df_cat_rate, on='rate_code_id', how='left')
    .join(df_cat_payment, on='payment_type_code', how='left')
    .join(df_cat_trip, on='trip_type_code', how='left')
    .join(df_cat_saf, on='store_and_fwd_flag_norm', how='left')
    .withColumn('trip_duration_minutes', (F.unix_timestamp('dropoff_datetime_utc') - F.unix_timestamp('pickup_datetime_utc')) / 60.0)
    .withColumn('trip_date', F.to_date('pickup_datetime_utc'))
)

In [9]:
# Publicacion de tablas curadas
curated_preactions = f'CREATE SCHEMA IF NOT EXISTS {curated_schema}'

write_snowflake_table(df_zones, f'{curated_schema}.DIM_TAXI_ZONES', preactions=curated_preactions)
write_snowflake_table(df_cat_vendor, f'{curated_schema}.DIM_VENDOR', preactions=curated_preactions)
write_snowflake_table(df_cat_rate, f'{curated_schema}.DIM_RATE_CODE', preactions=curated_preactions)
write_snowflake_table(df_cat_payment, f'{curated_schema}.DIM_PAYMENT_TYPE', preactions=curated_preactions)
write_snowflake_table(df_cat_trip, f'{curated_schema}.DIM_TRIP_TYPE', preactions=curated_preactions)
write_snowflake_table(df_cat_saf, f'{curated_schema}.DIM_STORE_AND_FWD', preactions=curated_preactions)
write_snowflake_table(df_trips_enriched, f'{curated_schema}.FCT_TRIPS_ENRICHED', preactions=curated_preactions)

In [ ]:
# Validaciones rapidas
df_trips_enriched.groupBy('taxi_type').count().orderBy('taxi_type').show()

df_trips_enriched.select(
    'trip_nk', 'taxi_type',
    'pickup_datetime_utc', 'dropoff_datetime_utc',
    'pulocationid', 'pu_borough', 'pu_zone',
    'dolocationid', 'do_borough', 'do_zone',
    'vendor_id', 'vendor_desc',
    'rate_code_id', 'rate_code_desc',
    'payment_type_code', 'payment_type_desc',
    'trip_type_code', 'trip_type_desc',
    'store_and_fwd_flag_norm', 'store_and_fwd_desc',
    'total_amount', 'trip_duration_minutes'
).show(20, truncate=False)

+---------+--------+
|taxi_type|   count|
+---------+--------+
|    green| 3083323|
|   yellow|25183429|
+---------+--------+

